# NanoQuant Qwen3.8-27B · inference and agent checks

[Открыть этот ноутбук в Google Colab](https://colab.research.google.com/github/rkuznetsovcg/NanoQuant/blob/main/notebooks/qwen38_nanoquant_agent_bench.ipynb)

Этот ноутбук использует **уже готовый NanoQuant `.pt` checkpoint** из Google Drive. Квантование и BF16-модель не запускаются. Он сравнивает обычный путь NanoQuant с `gemv`, `gemlite` и `gemm` на одной GPU, сохраняет скорость и время подготовки, затем поднимает OpenAI-compatible API для AppWorld, BenchLocal и HTML-примера.

Запускай ячейки по порядку. Для теста пропускной способности каждая версия загружает этот же checkpoint в отдельном процессе; так подготовка одного CUDA-ядра не меняет следующий вариант. Основное сравнение фиксирует входные 512 токенов, генерацию до 128 токенов, 2 разогрева и 7 измерений.

Checkpoint содержит текстовую часть Qwen3.8. Поэтому AppWorld, ToolCall-15 и генерация HTML доступны как текстовые проверки. Результаты мультимодальных строк таблицы вроде OSWorld, WebArena, AndroidWorld, SWE-MM и Vision2Web этим checkpoint не измеряются. HTML-пример ниже — демонстрация генерации HTML текстовой моделью, не официальный Vision2Web балл.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

MODEL_ID = "Qwen/Qwen3.8-27B"
MODEL_API_ID = "nanoquant-qwen38-27b"
RUN_NAME = "qwen38-27b-0.55-balanced"
CHECKPOINT_NAME = "Qwen3.8-27B-NQ-0.55-balanced.pt"
REPO_URL = "https://github.com/rkuznetsovcg/NanoQuant.git"
APPWORLD_URL = "https://github.com/StonyBrookNLP/appworld.git"

DRIVE_ROOT = Path("/content/drive/MyDrive/NanoQuant/runs") / RUN_NAME
DRIVE_CHECKPOINT = DRIVE_ROOT / CHECKPOINT_NAME
DRIVE_METADATA = DRIVE_ROOT / "run_complete.json"
LOCAL_ROOT = Path("/content/nanoquant-agent-bench")
LOCAL_CHECKPOINT = LOCAL_ROOT / CHECKPOINT_NAME
REPO_DIR = Path("/content/NanoQuant-agent-bench")
KERNEL_DIR = REPO_DIR / "src/nanoquant/kernel"
LOCAL_BENCH_DIR = LOCAL_ROOT / "results"
DRIVE_BENCH_DIR = DRIVE_ROOT / "agent-bench"

PROMPT_TOKENS = 512
MAX_NEW_TOKENS = 128
WARMUP_RUNS = 2
MEASURE_RUNS = 7
BENCH_KERNELS = ("none", "gemv", "gemlite", "gemm")

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_BENCH_DIR.mkdir(parents=True, exist_ok=True)
print("Настройки заданы. Следующая ячейка подключит Drive и найдёт checkpoint.")

## 1. Подключить Drive и проверить checkpoint

Ожидаемый файл лежит в `MyDrive/NanoQuant/runs/qwen38-27b-0.55-balanced/` рядом с `run_complete.json`.

In [ ]:
from google.colab import drive
from IPython.display import display, HTML
import torch

drive.mount("/content/drive")
DRIVE_BENCH_DIR.mkdir(parents=True, exist_ok=True)
if not DRIVE_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Не найден checkpoint: {DRIVE_CHECKPOINT}")
if not DRIVE_METADATA.is_file():
    raise FileNotFoundError(f"Не найден run_complete.json: {DRIVE_METADATA}")

run_metadata = json.loads(DRIVE_METADATA.read_text())
expected_bytes = int(run_metadata.get("checkpoint_size_bytes", 0))
actual_bytes = DRIVE_CHECKPOINT.stat().st_size
if expected_bytes and actual_bytes != expected_bytes:
    raise ValueError(f"Размер checkpoint не совпал с run_complete.json: {actual_bytes} != {expected_bytes}")
if run_metadata.get("model_id") not in (None, MODEL_ID):
    raise ValueError(f"В metadata указан другой model_id: {run_metadata.get('model_id')}")

if not LOCAL_CHECKPOINT.exists() or LOCAL_CHECKPOINT.stat().st_size != actual_bytes:
    print(f"Копирую {actual_bytes / 1024**3:.2f} GiB из Drive на локальный диск Colab…")
    shutil.copy2(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)

if not torch.cuda.is_available():
    raise RuntimeError("В Colab выбери среду выполнения с NVIDIA GPU.")
props = torch.cuda.get_device_properties(0)
print(f"Checkpoint: {LOCAL_CHECKPOINT} ({LOCAL_CHECKPOINT.stat().st_size / 1024**3:.2f} GiB)")
print(f"GPU: {props.name}; VRAM: {props.total_memory / 1024**3:.1f} GiB")

## 2. Подготовить отдельную копию кода и зависимости

Клонируется форк в отдельную папку Colab. Установка использует текущий PyTorch Colab; NanoQuant ставится с fast-зависимостями для линейного attention Qwen3.8.

In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)

install_log = LOCAL_ROOT / "install-nanoquant.log"
install_cmd = [sys.executable, "-m", "pip", "install", "-e", ".[qwen-fast]", "--no-build-isolation"]
with install_log.open("w") as log_file:
    result = subprocess.run(install_cmd, cwd=REPO_DIR, stdout=log_file, stderr=subprocess.STDOUT)
if result.returncode:
    print(install_log.read_text(errors="replace")[-12000:])
    raise RuntimeError("Установка NanoQuant/qwen-fast завершилась ошибкой.")
subprocess.run([sys.executable, "-m", "pip", "install", "fastapi", "uvicorn"], check=True)
print("NanoQuant, Qwen fast-attention зависимости, FastAPI и Uvicorn готовы.")

## 3. Подготовить AppWorld отдельно от PyTorch среды

AppWorld запускается в отдельном виртуальном окружении, чтобы его зависимости не обновляли PyTorch и CUDA-пакеты Colab, обслуживающие checkpoint. В конце установки регистрируется OpenAI-compatible endpoint локального сервера.

In [ ]:
APPWORLD_REPO = Path("/content/appworld")
APPWORLD_VENV = Path("/content/appworld-venv")
APPWORLD_ROOT = Path("/content/appworld-data")
APPWORLD_ROOT.mkdir(parents=True, exist_ok=True)
APPWORLD_READY = False
APPWORLD_SETUP_LOG = LOCAL_ROOT / "install-appworld.log"

try:
    if not APPWORLD_REPO.exists():
        subprocess.run(["git", "lfs", "install", "--skip-repo"], check=False)
        subprocess.run(["git", "clone", "--depth", "1", APPWORLD_URL, str(APPWORLD_REPO)], check=True)
        subprocess.run(["git", "lfs", "pull"], cwd=APPWORLD_REPO, check=True)
    if not APPWORLD_VENV.exists():
        subprocess.run([sys.executable, "-m", "venv", str(APPWORLD_VENV)], check=True)
    APPWORLD_PYTHON = APPWORLD_VENV / "bin/python"
    APPWORLD_CLI = APPWORLD_VENV / "bin/appworld"
    appworld_env = os.environ.copy()
    appworld_env["APPWORLD_ROOT"] = str(APPWORLD_ROOT)

    commands = [
        ([str(APPWORLD_PYTHON), "-m", "pip", "install", "-e", "."], APPWORLD_REPO),
        ([str(APPWORLD_PYTHON), "-m", "pip", "install", "-e", "experiments[simplified]"], APPWORLD_REPO),
        ([str(APPWORLD_CLI), "install", "--repo"], APPWORLD_REPO),
        ([str(APPWORLD_CLI), "download", "data"], APPWORLD_REPO),
    ]
    with APPWORLD_SETUP_LOG.open("w") as log_file:
        for command, cwd in commands:
            print("Запускаю:", " ".join(command))
            result = subprocess.run(command, cwd=cwd, env=appworld_env, stdout=log_file, stderr=subprocess.STDOUT)
            if result.returncode:
                print(APPWORLD_SETUP_LOG.read_text(errors="replace")[-12000:])
                raise RuntimeError("Подготовка AppWorld завершилась ошибкой; подробности выше.")

    model_registry = APPWORLD_REPO / "experiments/configs/_generator/models/nanoquant_qwen38_nq055.py"
    model_registry.write_text(
        "MODEL_INFOS = (\n"
        "    {\n"
        "        'model_name': 'qwen38-27b-nq-055',\n"
        "        'client_name': 'openai',\n"
        "        'model_id': '" + MODEL_API_ID + "',\n"
        "        'model_kwargs': {\n"
        "            'api_type': 'chat_completions',\n"
        "            'cost_per_token': {'input_cache_miss': 0.0, 'input_cache_hit': 0.0, 'input_cache_write': None, 'output': 0.0},\n"
        "            'tool_parser_name': None,\n"
        "            'drop_reasoning_content': False,\n"
        "            'api_key_env_name': 'NANOQUANT_API_KEY',\n"
        "            'base_url': 'http://127.0.0.1:8000/v1',\n"
        "            'temperature': 1.0,\n"
        "            'seed': 100,\n"
        "            'parallel_tool_calls': True,\n"
        "            'max_completion_tokens': 4096,\n"
            "            'reasoning_effort': None,\n"
        "            'stop': None,\n"
        "            'thinking_budget': None,\n"
            "        },\n"
        "        'function_calling': True,\n"
        "        'tool_choice': 'auto',\n"
        "        'function_calling_demos': True,\n"
        "        'part_of': ['v1', 'vllm'],\n"
        "        'provider': 'vllm',\n"
        "        'model_server_config': None,\n"
        "    },\n"
        ")\n",
        encoding="utf-8",
    )
    APPWORLD_READY = True
    print(f"AppWorld готов; рабочие данные: {APPWORLD_ROOT}")
    print(f"Лог установки: {APPWORLD_SETUP_LOG}")
except Exception as error:
    APPWORLD_READY = False
    print(f"AppWorld пока не подготовлен: {type(error).__name__}: {error}")
    if APPWORLD_SETUP_LOG.exists():
        print(APPWORLD_SETUP_LOG.read_text(errors="replace")[-12000:])
    print("Остальные ячейки бенчмарка можно продолжать; ячейка AppWorld сообщит, если запуск недоступен.")


## 4. Собрать доступные CUDA-ядра

`gemv` и `gemm` используют расширение из `src/nanoquant/kernel`. Компиляция идёт для текущей GPU Colab. Если компилятор или конкретное ядро не поддерживает GPU, тест этой опции ниже запишется как неудачный и продолжится со следующей.

In [ ]:
KERNEL_BUILD_LOG = LOCAL_ROOT / "compile-kernels.log"
compile_result = subprocess.run(
    ["bash", "compile_kernel.sh"],
    cwd=KERNEL_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    errors="replace",
)
KERNEL_BUILD_LOG.write_text(compile_result.stdout, encoding="utf-8")
print(compile_result.stdout[-10000:])
if compile_result.returncode:
    print("Native CUDA build не прошёл; gemlite и no-kernel всё равно будут измерены.")

## 5. Сравнить скорости и цену холодного старта

Все варианты получают одинаковые токены, dtype, prompt length, batch size и generation settings. На каждую опцию запускается новый процесс. В CSV записываются отдельные замеры decode tokens/s и памяти; лог фиксирует полную длительность, загрузку checkpoint и суммарную подготовку CUDA-ядер.

In [ ]:
import csv
import re
import statistics
import ipywidgets as widgets
from IPython.display import display

PYTHON_RUNNER = r'''
import runpy, sys, time
from nanoquant.modules.linear import NanoQuantLinear
stats = {"seconds": 0.0, "layers": 0}
original = NanoQuantLinear._prepare_kernel
def timed_prepare(self, *args, **kwargs):
    started = time.perf_counter()
    try:
        return original(self, *args, **kwargs)
    finally:
        stats["seconds"] += time.perf_counter() - started
        stats["layers"] += 1
NanoQuantLinear._prepare_kernel = timed_prepare
sys.argv = ["nanoquant.kernel.test_decode", *sys.argv[1:]]
try:
    runpy.run_module("nanoquant.kernel.test_decode", run_name="__main__")
finally:
    print(f"NQ_KERNEL_PREP_SECONDS={stats['seconds']:.6f} NQ_KERNEL_PREP_LAYERS={stats['layers']}", flush=True)
'''

measurements = []
iter_re = re.compile(r"\[iter\s+(\d+)/(\d+)\]")
load_re = re.compile(r"successfully loaded to .* in ([0-9.]+)s")
prep_re = re.compile(r"NQ_KERNEL_PREP_SECONDS=([0-9.]+) NQ_KERNEL_PREP_LAYERS=(\d+)")

for kernel_name in BENCH_KERNELS:
    csv_path = LOCAL_BENCH_DIR / f"decode-{kernel_name}.csv"
    log_path = LOCAL_BENCH_DIR / f"decode-{kernel_name}.log"
    command = [
        sys.executable, "-c", PYTHON_RUNNER,
        "--model_name", MODEL_ID,
        "--qmodel_ckpt", str(LOCAL_CHECKPOINT),
        "--use_quant_kernels", "True" if kernel_name != "none" else "False",
        "--quant_kernel_type", "gemv" if kernel_name == "none" else kernel_name,
        "--bench_prompt", "Give a short, direct answer to the user's request.",
        "--prompt_tokens", str(PROMPT_TOKENS),
        "--max_new_tokens", str(MAX_NEW_TOKENS),
        "--batch_size", "1",
        "--warmup", str(WARMUP_RUNS),
        "--iters", str(MEASURE_RUNS),
        "--output_csv", str(csv_path),
        "--gpu_id", "0",
        "--seed", "0",
        "--seqlen", "4096",
        "--dtype", "bfloat16",
        "--top_k", "20",
        "--temperature", "0.0",
        "--ppl_task", "",
        "--no_compile",
        "--profile_gpu_memory",
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
    env["PYTHONUNBUFFERED"] = "1"
    bar = widgets.IntProgress(value=0, min=0, max=MEASURE_RUNS, description=kernel_name, bar_style="info")
    status = widgets.HTML(value="Запускаю чистый процесс…")
    display(widgets.VBox([status, bar]))
    started = time.monotonic()
    output_lines = []
    process = subprocess.Popen(
        command,
        cwd=KERNEL_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        bufsize=1,
    )
    for line in process.stdout:
        output_lines.append(line)
        match = iter_re.search(line)
        if match:
            bar.value = min(MEASURE_RUNS, int(match.group(1)))
            status.value = f"{kernel_name}: измерение {match.group(1)}/{match.group(2)}"
        elif "Preparing kernels" in line or "Using custom CUDA" in line:
            status.value = f"{kernel_name}: пакую веса в формат ядра…"
    return_code = process.wait()
    elapsed = time.monotonic() - started
    raw_log = "".join(output_lines)
    log_path.write_text(raw_log, encoding="utf-8")

    row = {
        "kernel": kernel_name,
        "status": "ok" if return_code == 0 else "failed",
        "process_seconds": round(elapsed, 3),
        "checkpoint_load_seconds": None,
        "kernel_prepare_seconds": 0.0,
        "kernel_prepare_layers": 0,
        "median_decode_tokens_s": None,
        "measurements_decode_tokens_s": [],
        "csv": str(csv_path),
        "log": str(log_path),
        "error_tail": [],
    }
    load_match = load_re.search(raw_log)
    if load_match:
        row["checkpoint_load_seconds"] = float(load_match.group(1))
    prep_match = prep_re.search(raw_log)
    if prep_match:
        row["kernel_prepare_seconds"] = float(prep_match.group(1))
        row["kernel_prepare_layers"] = int(prep_match.group(2))
    if return_code == 0 and csv_path.exists():
        with csv_path.open(newline="", encoding="utf-8") as csv_file:
            for record in csv.DictReader(csv_file):
                if record.get("iter", "").isdigit() and record.get("decode_tps_total"):
                    row["measurements_decode_tokens_s"].append(float(record["decode_tps_total"]))
        if row["measurements_decode_tokens_s"]:
            row["median_decode_tokens_s"] = statistics.median(row["measurements_decode_tokens_s"])
        else:
            row["status"] = "failed"
            row["error_tail"] = ["CSV содержит ноль измерений."]
    else:
        row["error_tail"] = raw_log.splitlines()[-20:]
    if row["status"] == "failed":
        bar.bar_style = "danger"
        status.value = f"{kernel_name}: ошибка; смотри {log_path.name}"
        print("\n".join(row["error_tail"][-12:]))
    else:
        bar.value = MEASURE_RUNS
        bar.bar_style = "success"
        status.value = (
            f"{kernel_name}: median {row['median_decode_tokens_s']:.2f} токенов/с; "
            f"подготовка {row['kernel_prepare_seconds']:.1f} с; процесс {row['process_seconds']:.1f} с"
        )
    measurements.append(row)
    del process

successful_kernels = [r for r in measurements if r["status"] == "ok" and r["median_decode_tokens_s"] is not None]
if not successful_kernels:
    raise RuntimeError("Не прошёл ни один вариант инференса. Смотри логи в LOCAL_BENCH_DIR.")
optimized = [r for r in successful_kernels if r["kernel"] != "none"]
selected = max(optimized or successful_kernels, key=lambda r: r["median_decode_tokens_s"])
benchmark_result = {
    "model_id": MODEL_ID,
    "checkpoint": str(DRIVE_CHECKPOINT),
    "checkpoint_size_bytes": DRIVE_CHECKPOINT.stat().st_size,
    "gpu": torch.cuda.get_device_name(0),
    "dtype": "bfloat16",
    "prompt_tokens": PROMPT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": 1,
    "warmup_runs": WARMUP_RUNS,
    "measurement_runs": MEASURE_RUNS,
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "selected_kernel": selected["kernel"],
    "selection_rule": "fastest successful optimized kernel; fallback only if every optimized candidate failed",
    "results": measurements,
}
LOCAL_BENCH_DIR.joinpath("kernel_benchmark.json").write_text(json.dumps(benchmark_result, indent=2), encoding="utf-8")
shutil.copytree(LOCAL_BENCH_DIR, DRIVE_BENCH_DIR, dirs_exist_ok=True)

import pandas as pd
summary_rows = [
    {
        "kernel": r["kernel"],
        "median tok/s": r["median_decode_tokens_s"],
        "prep s": r["kernel_prepare_seconds"],
        "checkpoint load s": r["checkpoint_load_seconds"],
        "cold run s": r["process_seconds"],
        "status": r["status"],
    }
    for r in measurements
]
display(pd.DataFrame(summary_rows))
print(f"Выбранный kernel: {selected['kernel']}")
print(f"Результаты и логи сохранены в {DRIVE_BENCH_DIR}")

## 6. Поднять API с выбранным kernel

Сервер загружает этот checkpoint один раз, перемещает модель на GPU и один раз подготавливает выбранный kernel. После появления ответа на `/health` сервер готов к AppWorld и BenchLocal.

In [ ]:
import secrets
import requests
import ipywidgets as widgets
from IPython.display import display

if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    SERVER_PROCESS.wait(timeout=20)

API_KEY = secrets.token_urlsafe(32)
SERVER_LOG = DRIVE_BENCH_DIR / "openai-server.log"
server_env = os.environ.copy()
server_env["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + server_env.get("PYTHONPATH", "")
server_log_handle = SERVER_LOG.open("w")
server_command = [
    sys.executable, "-m", "nanoquant.openai_server",
    "--checkpoint", str(LOCAL_CHECKPOINT),
    "--model-id", MODEL_ID,
    "--model-name", MODEL_API_ID,
    "--kernel", selected["kernel"],
    "--dtype", "bfloat16",
    "--max-tokens", "8192",
    "--api-key", API_KEY,
    "--host", "127.0.0.1",
    "--port", "8000",
]
SERVER_PROCESS = subprocess.Popen(
    server_command,
    cwd=REPO_DIR,
    env=server_env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)
health_bar = widgets.IntProgress(value=0, min=0, max=600, description="API startup", bar_style="info")
health_status = widgets.HTML(value="Загружаю checkpoint и готовлю kernel…")
display(widgets.VBox([health_status, health_bar]))
health = None
for second in range(600):
    if SERVER_PROCESS.poll() is not None:
        break
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            headers={"Authorization": f"Bearer {API_KEY}"},
            timeout=3,
        )
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    health_bar.value = second + 1
    if second % 10 == 0:
        health_status.value = f"Сервер готовит модель… прошло {second} с"
    time.sleep(1)
if health is None:
    server_log_handle.flush()
    server_log_handle.close()
    print(SERVER_LOG.read_text(errors="replace")[-12000:])
    raise RuntimeError("NanoQuant API не поднялся; полный лог сохранён в Drive.")
health_bar.bar_style = "success"
health_status.value = f"API готов: {health['kernel']} на {health['device']}"
print(json.dumps(health, indent=2, ensure_ascii=False))
print("API key для BenchLocal:", API_KEY)

## 7. Открыть API для BenchLocal на Mac

BenchLocal — отдельное desktop-приложение. В нём создай OpenAI-compatible provider и подключи временный URL из следующей ячейки:

- Base URL: `https://<trycloudflare-url>/v1`
- API key: значение `API_KEY`, напечатанное при старте сервера
- Model ID: `nanoquant-qwen38-27b`
- Установи официальный bench pack **ToolCall-15** и запусти его с этой моделью. Для воспроизводимого теста выставь temperature 0.

Временный URL работает только пока Colab runtime и tunnel-процесс запущены; каждому запросу нужен bearer API key. Останови туннель после BenchLocal.

In [ ]:
import platform
import re
import requests
from IPython.display import display, HTML

if platform.machine() not in ("x86_64", "amd64"):
    raise RuntimeError(f"Для Colab ожидается Linux x86_64, получено: {platform.machine()}")
CLOUDFLARED = Path("/content/cloudflared")
if not CLOUDFLARED.exists():
    download = subprocess.run(
        ["wget", "-q", "-O", str(CLOUDFLARED), "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        capture_output=True,
        text=True,
    )
    if download.returncode:
        raise RuntimeError(download.stderr or "Не удалось скачать cloudflared.")
    CLOUDFLARED.chmod(0o755)

TUNNEL_LOG = DRIVE_BENCH_DIR / "cloudflared.log"
tunnel_log_handle = TUNNEL_LOG.open("w")
TUNNEL_PROCESS = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log_handle,
    stderr=subprocess.STDOUT,
    text=True,
)
tunnel_bar = widgets.IntProgress(value=0, min=0, max=90, description="Tunnel", bar_style="info")
tunnel_status = widgets.HTML(value="Открываю временный endpoint…")
display(widgets.VBox([tunnel_status, tunnel_bar]))
TUNNEL_URL = None
for second in range(90):
    if TUNNEL_PROCESS.poll() is not None:
        break
    tunnel_log_handle.flush()
    raw = TUNNEL_LOG.read_text(errors="replace")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", raw)
    if match:
        TUNNEL_URL = match.group(0)
        break
    tunnel_bar.value = second + 1
    time.sleep(1)
if not TUNNEL_URL:
    tunnel_log_handle.close()
    print(TUNNEL_LOG.read_text(errors="replace")[-8000:])
    raise RuntimeError("Cloudflare Quick Tunnel не выдал URL.")
probe = requests.get(
    TUNNEL_URL + "/health",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=15,
)
probe.raise_for_status()
tunnel_bar.bar_style = "success"
tunnel_status.value = "BenchLocal endpoint отвечает."
print("Base URL:", TUNNEL_URL + "/v1")
print("Model ID:", MODEL_API_ID)
print("API key:", API_KEY)

## 8. Запустить AppWorld на этой модели

По умолчанию запускается только официальный split `test_normal`. Не открывай per-task результаты тестового split; сохраняем и показываем только итоговые числовые метрики. Полный лог CLI остаётся локально в Colab.

In [ ]:
import json
import re
import time
import requests
from IPython.display import display
import ipywidgets as widgets

APPWORLD_DATASETS = ("test_normal",)
if not APPWORLD_READY:
    raise RuntimeError("Сначала успешно выполни ячейку подготовки AppWorld.")
if SERVER_PROCESS.poll() is not None:
    raise RuntimeError("NanoQuant API остановлен. Выполни ячейку запуска сервера.")

os.environ["NANOQUANT_API_KEY"] = API_KEY
appworld_env["NANOQUANT_API_KEY"] = API_KEY
appworld_env["APPWORLD_ROOT"] = str(APPWORLD_ROOT)

def aggregate_numbers(value, prefix=""):
    numbers = {}
    if isinstance(value, dict):
        for key, child in value.items():
            key_text = str(key).lower()
            if "task" in key_text or "trace" in key_text or "example" in key_text:
                continue
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(child, (int, float)) and any(
                word in key_text for word in ("score", "success", "pass", "accuracy", "rate", "total", "count")
            ):
                numbers[child_prefix] = child
            elif isinstance(child, dict):
                numbers.update(aggregate_numbers(child, child_prefix))
    return numbers

appworld_summaries = []
for dataset_name in APPWORLD_DATASETS:
    log_path = DRIVE_BENCH_DIR / f"appworld-{dataset_name}.log"
    command = [
        str(APPWORLD_CLI), "run", "auto",
        "--agent-name", "simplified_function_calling_agent",
        "--model-name", "qwen38-27b-nq-055",
        "--dataset-name", dataset_name,
        "--with-evaluation",
        "--num-processes", "1",
        "--root", str(APPWORLD_ROOT),
    ]
    status = widgets.HTML(value="AppWorld выполняет задачи; логи задач не выводятся в ячейку.")
    display(status)
    started = time.monotonic()
    with log_path.open("w") as log_file:
        process = subprocess.Popen(
            command,
            cwd=APPWORLD_REPO,
            env=appworld_env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )
        while process.poll() is None:
            elapsed_live = time.monotonic() - started
            SERVER_LOG.touch(exist_ok=True)
            server_log_handle.flush()
            completed_calls = sum(
                '"POST /v1/chat/completions ' in line and ' 200 ' in line
                for line in SERVER_LOG.read_text(errors="replace").splitlines()
            )
            status.value = (
                f"{dataset_name}: идёт · {elapsed_live / 60:.1f} мин · "
                f"NanoQuant обработал {completed_calls} запросов; отдельные task traces скрыты"
            )
            time.sleep(5)
        return_code = process.wait()
    elapsed = time.monotonic() - started
    if return_code:
        status.value = f"{dataset_name}: ошибка; смотри лог в Drive"
        print(log_path.read_text(errors="replace")[-8000:])
        raise RuntimeError(f"AppWorld run завершился с кодом {return_code}.")

    candidates = list((APPWORLD_ROOT / "experiments/outputs").rglob(f"{dataset_name}.json"))
    if not candidates:
        status.value = f"{dataset_name}: отчёт не найден"
        raise FileNotFoundError(f"Не найден итоговый {dataset_name}.json внутри {APPWORLD_ROOT}")
    report_path = max(candidates, key=lambda p: p.stat().st_mtime)
    report = json.loads(report_path.read_text())
    summary = {
        "dataset": dataset_name,
        "elapsed_seconds": round(elapsed, 2),
        "aggregate_metrics": aggregate_numbers(report),
    }
    appworld_summaries.append(summary)
    summary_path = DRIVE_BENCH_DIR / f"appworld-{dataset_name}-aggregate.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    status.value = f"{dataset_name} завершён за {elapsed / 60:.1f} мин"
    print(json.dumps(summary, indent=2, ensure_ascii=False))

## 9. Проверить генерацию HTML

Сервер получает исходный текстовый prompt, сохраняет сгенерированный однофайловый HTML в Drive и показывает изолированный preview. Это качественная демонстрация вывода, отдельная от официального Vision2Web набора.

In [ ]:
import html
import requests
import concurrent.futures
import ipywidgets as widgets
from IPython.display import HTML, display

HTML_PROMPT = (
    "Design a richly crafted voxel-art environment featuring an ornate pagoda set within a vibrant garden "
    "as a single self-contained HTML file. Return only the complete HTML document. Use embedded CSS and JavaScript, "
    "no external assets, and make the scene interactive with mouse-or-touch camera controls."
)
html_status = widgets.HTML(value="Модель пишет HTML…")
display(html_status)
request_started = time.monotonic()
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    future = executor.submit(
        requests.post,
        "http://127.0.0.1:8000/v1/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "model": MODEL_API_ID,
            "messages": [{"role": "user", "content": HTML_PROMPT}],
            "temperature": 0.7,
            "top_p": 0.95,
            "max_completion_tokens": 4096,
        },
        timeout=900,
    )
    while not future.done():
        html_status.value = f"Модель пишет HTML… прошло {(time.monotonic() - request_started) / 60:.1f} мин"
        time.sleep(2)
    response = future.result()
response.raise_for_status()
html_status.value = f"Ответ получен за {(time.monotonic() - request_started) / 60:.1f} мин"
answer = response.json()["choices"][0]["message"].get("content") or ""
answer = answer.strip()
if answer.startswith("```html"):
    answer = answer[7:]
    if answer.endswith("```"):
        answer = answer[:-3]
elif answer.startswith("```"):
    answer = answer[3:]
    if answer.endswith("```"):
        answer = answer[:-3]
answer = answer.strip()
if "<html" not in answer.lower() and "<!doctype html" not in answer.lower():
    raise RuntimeError("Модель не вернула HTML документ; ответ сохранён в ответе API ячейки.")

HTML_OUTPUT = DRIVE_BENCH_DIR / "qwen38-voxel-pagoda.html"
HTML_OUTPUT.write_text(answer, encoding="utf-8")
print(f"Сохранено: {HTML_OUTPUT}; {len(answer):,} символов")
iframe_src = html.escape(answer, quote=True)
display(HTML(
    '<iframe sandbox="allow-scripts" referrerpolicy="no-referrer" '
    'style="width:100%;height:760px;border:1px solid #888;border-radius:8px" '
    f'srcdoc="{iframe_src}"></iframe>'
))

## 10. Остановить публичный URL и модель

Выполни после BenchLocal и AppWorld. Убийство процессов завершает туннель и освобождает GPU.

In [ ]:
for process_name in ("TUNNEL_PROCESS", "SERVER_PROCESS"):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=10)
for handle_name in ("tunnel_log_handle", "server_log_handle"):
    handle = globals().get(handle_name)
    if handle is not None and not handle.closed:
        handle.close()
print("NanoQuant API и временный tunnel остановлены.")